# FinBot Fine-Tuning Notebook

DART 기반 재무 Q&A 데이터로 `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` 모델을 코랩에서 LoRA 파인튜닝하는 노트북입니다.

## 1. 라이브러리 설치

코랩 런타임에서 필요한 라이브러리를 설치합니다.

In [11]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

## 2. 모델 로드

구글 드라이브를 마운트하고, 4bit 양자화 설정으로 DeepSeek 계열 경량 모델을 로드합니다.

In [12]:
import os
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')

MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
TRAIN_PATH = '/content/drive/MyDrive/2026/train.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/finbot-model'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

## 3. 데이터 로드

구글 드라이브에 올려둔 `train.jsonl`을 읽고, Alpaca 스타일 프롬프트 템플릿으로 학습용 텍스트를 만듭니다.

In [13]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')

def format_example(example):
    prompt = (
        '### Instruction:\n'
        f"{example['instruction']}\n\n"
        '### Input:\n'
        f"{example['input']}\n\n"
        '### Response:\n'
        f"{example['output']}"
    )
    return {'text': prompt}

dataset = dataset.map(format_example)
dataset[0]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'instruction': '?? ????? ?? ??? ???.',
 'input': 'LG에너지솔루션의 2024년 매출은 얼마인가요?',
 'output': '연결회사의 2024년 연간 매출액은 약 25조 6,196억원입니다.',
 'text': '### Instruction:\n?? ????? ?? ??? ???.\n\n### Input:\nLG에너지솔루션의 2024년 매출은 얼마인가요?\n\n### Response:\n연결회사의 2024년 연간 매출액은 약 25조 6,196억원입니다.'}

## 4. LoRA 설정

코랩 무료 GPU에 맞춰 LoRA 어댑터를 설정합니다.

In [14]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
)

peft_config

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

## 5. 학습

`SFTTrainer`로 1 epoch만 가볍게 학습합니다. 무료 GPU 기준으로 배치 크기는 작게 유지합니다.

In [15]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/content/finbot-checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    max_seq_length=1024,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=lambda example: example['text'],
)

trainer.train()

TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'

## 6. 저장

학습된 어댑터와 토크나이저를 구글 드라이브에 저장합니다.

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to: {OUTPUT_DIR}')

## 7. 테스트

학습된 모델에 질문을 넣어 간단히 응답을 확인합니다.

In [ ]:
test_prompt = (
    '### Instruction:\n'
    '재무 분석가로서 다음 질문에 답하라.\n\n'
    '### Input:\n'
    '삼성전자 2024년 영업이익은?\n\n'
    '### Response:\n'
)

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))